# Silverbox: understanding the data and the prediction task

This notebook builds Task 5 step by step: data preparation, acceleration networks,
RK4 integration, training, and free-run evaluation. Explanations accompany each step.

### Running on Kaggle

1. Import `Duffing_PINN_Kaggle.ipynb` into a Kaggle Python notebook.
2. Attach a dataset containing **SNLS80mV.csv** and the supplied illustration PNG.
   Keep the original CSV filename. The loader searches under `/kaggle/input`, so the
   dataset folder name does not need to match a particular name.
3. To use a GPU, choose one under **Settings > Accelerator**. The code selects CUDA
   if available, otherwise CPU. It uses one GPU, even if the session offers two.
4. Run cells in order. The explicitly marked training cell trains all six trials (three MLP / physics-informed pairs).
   Running every cell also runs training; there is no separate hidden training command.
5. Completed trial models, histories, scores, and plots are written to a new run folder
   under `/kaggle/working/silverbox_results`. Use Save Version -> Save & Run All for a saved run,
   or download the checkpoint ZIP during an interactive session. Downloading only the
   notebook does not keep the learned weights. See the checkpoint section before training.

The code uses NumPy, pandas, Matplotlib, and PyTorch. It does not install packages or need
internet access once the data is attached. It uses ordinary float32 training, without
mixed precision or multi-GPU setup. Small sequential MLP calls may not be faster on a GPU;
`device_choice` can be set to `"cpu"` for a later timing comparison.

Environment references: [Kaggle Python environment](https://github.com/Kaggle/docker-python),
[GPU setup](https://docs.nvidia.com/datascience/deployment/latest/platforms/kaggle/).

## 1. Load the recorded voltages

The CSV has an empty extra column, so we explicitly read only `V1` and `V2`.
Locally, the loader also supports `SilverboxFiles/SNLS80mV.csv`.
If more than one matching Kaggle file is attached, set `DATA_PATH` to the exact file.

This revision compares three plain MLPs with three physics-informed MLPs, saves resumable
checkpoints every 50 updates, and exports predictions and error maps for every completed model.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATA_PATH = None  # Optional exact path, e.g. "/kaggle/input/my-dataset/SNLS80mV.csv"
kaggle_input_root = Path("/kaggle/input")
running_on_kaggle = kaggle_input_root.is_dir()

if DATA_PATH is not None:
    path = Path(DATA_PATH).expanduser()
elif running_on_kaggle:
    candidates = sorted(kaggle_input_root.rglob("SNLS80mV.csv"))
    if not candidates:
        raise FileNotFoundError(
            "Attach a Kaggle dataset containing the extracted SNLS80mV.csv file, then rerun this cell."
        )
    if len(candidates) > 1:
        raise ValueError(f"Multiple SNLS80mV.csv files found. Set DATA_PATH explicitly: {candidates}")
    path = candidates[0]
else:
    path = Path("SilverboxFiles") / "SNLS80mV.csv"

if not path.is_file():
    raise FileNotFoundError(f"CSV not found: {path}. Set DATA_PATH to its actual location.")

data = pd.read_csv(path, usecols=["V1", "V2"], dtype=np.float64)
if len(data) < 46746:
    raise ValueError("The file is too short for the required [42650:46746] excerpt.")
if not np.isfinite(data.iloc[42650:46746].to_numpy()).all():
    raise ValueError("The required excerpt contains missing or non-finite voltages.")

output_root = (Path("/kaggle/working") if running_on_kaggle else Path("output")) / "silverbox_results"
print(f"Loaded {len(data):,} rows from {path}")
print(f"Results will be written under {output_root}")
data.head()

## 2. What are we trying to do?

We have a recording of an electronic circuit:

- **V1 is input voltage:** what was applied to the circuit.
- **V2 is output voltage:** how the circuit responded.

We want to learn how the output voltage changes over time. Then we will simulate it,
using the known input voltage and our own previous predictions.


The circuit behaves like a moving object attached to a spring. This is an analogy:
we are measuring voltage, not the position of a real object.
Whenever the assignment says **position**, we will call it **output voltage** here.

- **Input:** pushes the object.
- **Damping:** slows its motion.
- **Spring:** pulls it back toward its resting position (the center).
- **Cubic effect:** adds a stronger pull when the object is farther from the center.
  This is part of the same nonlinear spring, not another object.

In [ ]:
from IPython.display import Image, display

image_filenames = ["silverbox-explanation.png", "ChatGPT Image Oct 11, 2026, 01_03_40 AM.png"]
image_candidates = [path.parent / name for name in image_filenames if (path.parent / name).is_file()]
if not image_candidates and running_on_kaggle:
    image_candidates = [item for name in image_filenames for item in kaggle_input_root.rglob(name)]
if not image_candidates:
    image_candidates = [Path(name) for name in image_filenames if Path(name).is_file()]

if image_candidates:
    display(Image(filename=str(image_candidates[0]), width=1000))
else:
    print("Optional illustration not found. Attach silverbox-explanation.png to display it.")

### The quantities we need

| Symbol | Meaning in this task | Units | Where it comes from |
|---|---|---|---|
| $u$ | Input voltage, after subtracting its mean | V | Recorded `V1` |
| $y$ | Output voltage, after subtracting its mean | V | Recorded `V2` for training; our simulation during a free run |
| $v=\dot y$ | Rate of change of output voltage | V/s | Estimated from training output for initialization/exploration; simulated afterwards |
| $\alpha=\dot v=\ddot y$ | Rate of change of $v$ (called acceleration) | V/s² | Predicted by the network |
| $\Delta t$ | Time between samples | s | $1/610.35$ |

A dot above a quantity means its rate of change over time. We do **not** calculate time from voltage.
The sampling frequency already gives us the time between measurements.

The physical equation is

$$\ddot y = gu-a\dot y-by-cy^3.$$

The circuit coefficients $a,b,c,g$ describe damping, linear stiffness, cubic stiffness,
and input gain. Their exact values are not provided. They are different from the network weights.
The network will learn an acceleration function that represents this behavior.

### What enters the network, and what comes out?

$$[y,\;v,\;u]\quad\xrightarrow{\text{network}}\quad\alpha_\theta(y,v,u).$$

The word **output** can be confusing:

- **Circuit output:** voltage $y$.
- **Network output:** acceleration $\alpha_\theta$.

Yes, the current output voltage is a network input. The voltage at a **later time** is
what we want the simulation to predict. These are values at different times.

### What does the integrator do?

The network gives acceleration. The integrator uses it to move the state $(y,v)$ forward in time:

$$\dot y=v,\qquad \dot v=\alpha_\theta(y,v,u).$$

**Integrating means adding up small changes over time.**

- Adding up acceleration over time gives a **change in the rate $v$**.
- Adding up the rate $v$ over time gives a **change in output voltage $y$**.

For intuition, a simple Euler step is

$$\hat y_{k+1}=\hat y_k+\hat v_k\Delta t,$$
$$\hat v_{k+1}=\hat v_k+\alpha_\theta(\hat y_k,\hat v_k,u_k)\Delta t.$$

A hat means a simulated value. RK4, suggested in the assignment, approximates the changes
more accurately; we will discuss it before implementing it.

### A small example: inputs, prediction, and label

These numbers are **made up**, and $\Delta t=0.01$ s here is only for easy arithmetic.
The real data uses $\Delta t=1/610.35$ s.

| Network input at the current time | Example value |
|---|---|
| Output voltage $y_0$ | 0.10 V |
| Rate of change $v_0$ | 0.20 V/s |
| Input voltage $u_0$ | 0.03 V |

Suppose the network receives `[0.10, 0.20, 0.03]` and predicts acceleration **1.50 V/s²**.
One Euler step gives

$$\hat y_1=0.10+0.20(0.01)=0.102\;\mathrm{V},$$
$$\hat v_1=0.20+1.50(0.01)=0.215\;\mathrm{V/s}.$$

If the measured training voltage at that next time is **0.104 V**, that is our **label (target)**.
We compare the simulated voltage 0.102 with the measured voltage 0.104.
The labels for the training approach we will build are measured **output voltages**, not accelerations.

One detail in this simple Euler example: the new acceleration changes $v_1$ first,
and affects voltage through that rate in later steps. So this first voltage error alone
cannot teach the acceleration network. Training on a sequence lets acceleration predictions
affect the voltage errors. RK4 also uses intermediate rates within each step.

If the next known input voltage is 0.04 V, the next network input is

$$[\underbrace{0.102}_{\text{simulated voltage}},\;
\underbrace{0.215}_{\text{simulated rate}},\;
\underbrace{0.04}_{\text{recorded input}}].$$

The measured 0.104 is used for the training error; it does not replace 0.102 in this free simulation.

### Where does the starting rate come from? What happens during testing?

We initialize once, at the **beginning of the training record**:

$$y_0=y_{\mathrm{train}}[0],\qquad
v_0\approx\frac{y_{\mathrm{train}}[1]-y_{\mathrm{train}}[0]}{\Delta t}.$$

Both measured output values are from training. The initial rate means
"change in output voltage divided by the time taken."

For the required final evaluation:

1. Start with this training voltage and estimated rate.
2. Simulate through the training interval using the recorded input $u$.
3. Continue into testing with the same simulated state. **Do not reset at the boundary.**
4. During testing, use the known input voltage and simulated $y,v$.
5. Use measured test output only afterwards to calculate error and draw the comparison plot.

**Do not calculate a test starting rate from measured test output.**

Errors can accumulate: a wrong acceleration changes the next rate and voltage, and these
values then enter the network. During training, we adjust the weights to reduce voltage errors
across a simulated sequence. A simulation step moves time forward; a training update changes weights.

Eventually we must compare multiple architectures and hyperparameters, report training and test
free-run RMSE, and plot predicted versus measured test output. RMSE measures voltage error in volts;
smaller is better. Predicting zero gives about 0.06 V according to the assignment,
not a score measured by this notebook.

## 3. Prepare the specified excerpt

Use rows `[42650:46746]`: 4096 samples. Subtract each voltage's own mean over **all 4096 samples**,
as explicitly required by the assignment. Then split in time: first 3072 for training,
last 1024 for testing. Do not shuffle.

Centering over the full excerpt is the assignment's preprocessing rule. After that,
measured test output is reserved for evaluation, not for constructing simulated states.

In [ ]:
sampling_frequency = 610.35  # samples per second (Hz)
dt = 1.0 / sampling_frequency  # seconds between samples
excerpt_start = 42650
excerpt_stop = 46746
training_sample_count = 3072

# Preserve the original file row indices for reference.
data_roi = data.iloc[excerpt_start:excerpt_stop].copy()

u_raw = data_roi["V1"].to_numpy(dtype=float)
y_raw = data_roi["V2"].to_numpy(dtype=float)
u_mean = u_raw.mean()
y_mean = y_raw.mean()


u = u_raw - u_mean
y = y_raw - y_mean
t = np.arange(len(y)) * dt  # time since the start of the excerpt

u_train = u[:training_sample_count].copy()
y_train = y[:training_sample_count].copy()
t_train = t[:training_sample_count].copy()

u_test = u[training_sample_count:].copy()  # known drive during testing
y_test = y[training_sample_count:].copy()  # evaluation targets only
t_test = t[training_sample_count:].copy()  

## 4. Estimate the rate using training data only

`v_train_est` estimates the rate of change of output voltage for exploration.
It is not directly measured. Finite differences can amplify measurement noise.
We apply `np.gradient` only to `y_train`, so its last estimate cannot read a test output sample.

`y0` and `v0` are the one-time initial state for the future whole-excerpt free run.
We use the first-two-training-samples formula for `v0`, matching the explanation above.
The full estimated rate array is for exploration; it will not be fed in as measured rates at
every step of a free run.

This finite-difference rate is an approximation, not ground truth. In particular, the
first rate uses only two samples and can be inaccurate for fast oscillations. The same
initial-state rule is used for every candidate so their scores remain comparable.
If the free run drifts, this initialization and the training rate estimates are things
to investigate, along with the network and optimizer; do not use test output to fix them.

In [ ]:
v_train_est = np.gradient(y_train, dt, edge_order=1)

y0 = float(y_train[0])
v0 = float((y_train[1] - y_train[0]) / dt)
initial_state = np.array([y0, v0], dtype=float)  # [output voltage, rate]

# This is what the future network receives at the very first step.
# There is no network or acceleration prediction yet.
first_network_input = np.array([y0, v0, u_train[0]], dtype=float)

# Measured voltages are the training targets for a future simulated trajectory.
y_train_targets = y_train.copy()

### Inspect the variables before building a model

The table below puts recorded training values beside the estimated rate.
A row is not a complete independent network training example: the later training loop will
simulate a sequence, then compare predicted voltages with the corresponding targets.
In a free run, only the first state is initialized from measurements; later states are simulated.

In [ ]:
training_preview = pd.DataFrame({
    "time [s]": t_train,
    "input u [V]": u_train,
    "target output y [V]": y_train_targets,
    "estimated rate v [V/s]": v_train_est,
})
training_preview.head()

In [ ]:
print(f"Time between samples: {dt:.8f} s")
print(f"Training samples: {len(y_train)}; test samples: {len(y_test)}")
print(f"Test starts at: {t_test[0]:.5f} s")
print("Initial state [y0, v0]:", initial_state)
print("First network input [y0, v0, u0]:", first_network_input)

## 5. Explore the prepared signals

These plots use the same centered variables defined above.
The time plot shows the full excerpt and its boundary. The statistics, spectrum, and
voltage-versus-rate plot below use training samples only.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, u, linewidth=0.8, color="tab:blue")
axes[0].set_ylabel("Input u [V]")
axes[0].set_title("Centered Silverbox input")

axes[1].plot(t, y, linewidth=0.8, color="tab:orange")
axes[1].set_ylabel("Output y [V]")
axes[1].set_xlabel("Time since excerpt start [s]")
axes[1].set_title("Centered measured output (test part is for evaluation)")

for axis in axes:
    axis.axvline(t_test[0], color="black", linestyle="--", label="Train-test boundary")
    axis.grid(alpha=0.3)
    axis.legend()

plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({"input u [V]": u_train, "output y [V]": y_train}).agg(
    ["mean", "std", "min", "max"]
)

### Training output in the frequency domain

The spectrum shows which oscillation frequencies are present. A Hann window reduces
edge effects in this finite training record. Each spectrum is normalized to its own largest
peak, so the vertical axis compares relative levels, not absolute voltage amplitudes.

In [ ]:
n = len(y_train)
frequencies = np.fft.rfftfreq(n, d=dt)

raw_fft = np.fft.rfft(y_train)
hann_window = np.hanning(n)
hann_fft = np.fft.rfft(y_train * hann_window)

raw_amplitude = np.abs(raw_fft) / n
hann_amplitude = np.abs(hann_fft) / hann_window.sum()
# Double interior bins for a one-sided spectrum; leave DC and Nyquist alone.
if n % 2 == 0:
    raw_amplitude[1:-1] *= 2
    hann_amplitude[1:-1] *= 2
else:
    raw_amplitude[1:] *= 2
    hann_amplitude[1:] *= 2

epsilon = 1e-12
raw_db = 20 * np.log10(np.maximum(raw_amplitude / raw_amplitude.max(), epsilon))
hann_db = 20 * np.log10(np.maximum(hann_amplitude / hann_amplitude.max(), epsilon))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, sharey=True)
axes[0].plot(frequencies, raw_db, linewidth=0.8, color="tab:blue")
axes[0].set_title("Training output spectrum without a window")
axes[1].plot(frequencies, hann_db, linewidth=0.8, color="tab:orange")
axes[1].set_title("Training output spectrum with a Hann window")
axes[1].set_xlabel("Frequency [Hz]")
for axis in axes:
    axis.set_ylabel("Relative magnitude [dB]")
    axis.set_xlim(0, 150)
    axis.set_ylim(-100, 5)
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Output voltage versus its rate of change

This is also called a phase portrait. The horizontal axis is **output voltage** (`y_train`),
not input voltage. The vertical axis is its estimated rate (`v_train_est`).
It shows why voltage alone is not enough to describe the state: at the same voltage,
the signal can be rising or falling. No test-output derivative is calculated.

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(y_train, v_train_est, linewidth=0.7, alpha=0.8, label="Training record")
plt.xlabel("Output voltage y [V]")
plt.ylabel("Estimated rate of change v [V/s]")
plt.title("Training output voltage versus its rate of change")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Scale the numbers seen by the network

We choose training standard deviations as the three input scales:

$$s_y=\operatorname{std}(y_{\mathrm{train}}),\qquad
s_v=\operatorname{std}(v_{\mathrm{train,est}}),\qquad
s_u=\operatorname{std}(u_{\mathrm{train}}).$$

The training rate is an estimate, so its scale is also approximate. We keep all three scales
fixed during a training run and reuse them for testing. We do not calculate scales from test data.

In [ ]:
input_scale_summary = pd.DataFrame({
    "y: output voltage [V]": y_train,
    "v: estimated rate [V/s]": v_train_est,
    "u: input voltage [V]": u_train,
}).agg(["min", "max", "mean"])

input_scale_summary.loc["std (ddof=0)"] = [
    np.std(y_train, ddof=0),
    np.std(v_train_est, ddof=0),
    np.std(u_train, ddof=0),
]
input_scale_summary

In [ ]:
y_scale = float(np.std(y_train, ddof=0))
v_scale = float(np.std(v_train_est, ddof=0))
u_scale = float(np.std(u_train, ddof=0))

# Order matches the future network's inputs: [y, v, u].
input_scales = np.array([y_scale, v_scale, u_scale], dtype=float)
if not np.all(np.isfinite(input_scales)) or np.any(input_scales <= 0):
    raise ValueError("Input scales must be finite and positive. Inspect the training data.")

### Keep the simulation in physical units

The simulation will store output voltage in V and its rate in V/s. Time stays in seconds.
We divide only when giving those values to the network:

$$[y,v,u]\ \longrightarrow\ [y/s_y,\ v/s_v,\ u/s_u].$$

**Made-up example:** if the actual values are `[0.10, 20, 0.03]` and the scales are
`[0.10, 40, 0.06]`, the network receives `[1.0, 0.5, 0.5]`.
The actual output voltage is still 0.10 V; we have not changed the physical state.

Why keep the integrator in physical units? Its equations remain easy to read:
$\dot y=v$ and $\dot v=\alpha$. If we integrated scaled states instead, extra scale ratios
would enter these equations. Both approaches are possible; this one is easier to follow.

The arrays below are for inspecting the scaling. During a free run, we will scale the
**current simulated** voltage and rate at every network call, rather than feeding in
the measured training trajectory. Recorded input voltage is still available throughout.

In [ ]:
# Keep all existing physical-unit arrays unchanged.
y_train_scaled = y_train / y_scale
v_train_est_scaled = v_train_est / v_scale
u_train_scaled = u_train / u_scale
u_test_scaled = u_test / u_scale  # reuse the training scale

first_network_input_scaled = first_network_input / input_scales

pd.DataFrame({
    "quantity": ["output voltage y", "rate v", "input voltage u"],
    "physical value at start": first_network_input,
    "divide by": input_scales,
    "network receives": first_network_input_scaled,
})

### Acceleration scale: a starting choice, not a measured law

We have no directly measured acceleration labels. We will try

$$s_\alpha=(2\pi f_0)^2s_y,$$

where $f_0$ is the strongest nonzero frequency in the windowed **training output** spectrum.
The assignment says the main oscillation is near 70 Hz. We estimate the peak from training
data rather than forcing the answer to be 70. A finite record gives an approximate frequency.

**Where does this formula come from?** Imagine one simple oscillation:

$$y(t)=A\sin(2\pi f_0t).$$

Its rate and acceleration are

$$v(t)=A(2\pi f_0)\cos(2\pi f_0t),$$
$$\alpha(t)=-A(2\pi f_0)^2\sin(2\pi f_0t)=-(2\pi f_0)^2y(t).$$

Each derivative brings in a factor of $2\pi f_0$. Two derivatives give its square.
For this simple signal, acceleration's size is voltage's size multiplied by that square.
The minus sign tells us direction; the scale itself is positive.

**Our real signal is not a single sine wave.** We are borrowing this relationship only to
choose a reasonable numerical size. We are not forcing the network to follow a sine wave,
and this does not set the circuit coefficients. We can reconsider the scale after training.

For example, **if** $s_y=0.06$ V and $f_0=70$ Hz, then $s_\alpha\approx11607$ V/s².
That large number is possible because the signal changes direction very quickly.
These are illustrative numbers, not results from running this notebook.

We could differentiate the measurements twice, but that can amplify noise strongly.
The frequency-based choice avoids needing that second numerical derivative.

In [ ]:
# Estimate the dominant frequency using training output only.
# Remove its remaining mean for this spectrum only; do not change y_train.
scale_fft_signal = y_train - y_train.mean()
scale_fft_window = np.hanning(len(scale_fft_signal))
scale_fft_magnitude = np.abs(np.fft.rfft(scale_fft_signal * scale_fft_window))
scale_fft_frequencies = np.fft.rfftfreq(len(scale_fft_signal), d=dt)

# Exclude the zero-frequency (constant) component.
dominant_frequency_index = 1 + int(np.argmax(scale_fft_magnitude[1:]))
dominant_frequency_hz = float(scale_fft_frequencies[dominant_frequency_index])
angular_frequency = 2.0 * np.pi * dominant_frequency_hz
acceleration_scale = float(angular_frequency**2 * y_scale)

print(f"Training spectrum peak: {dominant_frequency_hz:.3f} Hz")
print(f"Frequency-bin spacing: {scale_fft_frequencies[1]:.3f} Hz")
print(f"Acceleration scale: {acceleration_scale:.3f} V/s^2")

### Convert the network's answer back before integration

Let $q_\theta$ be the number returned by the future network. It has no physical units.
We convert it into physical acceleration by multiplying:

$$\alpha_\theta(y,v,u)=s_\alpha\,
q_\theta(y/s_y,\ v/s_v,\ u/s_u).$$

For example, a network answer of 0.5 means acceleration is $0.5s_\alpha$ V/s².
The network may return values larger than 1 or negative values. The scale is **not a limit**
and **not the learning rate**. It is just the conversion factor between two representations.

```text
Simulation stores y [V], v [V/s]; recorded u [V] is known
                       |
                 divide by s_y, s_v, s_u
                       |
              network receives [y/s_y, v/s_v, u/s_u]
                       |
              network returns scaled acceleration q
                       |
                 multiply by s_alpha
                       |
              integrator receives acceleration [V/s^2]
                       |
              update physical y and v; repeat
```

The training labels remain measured output voltages (`y_train_targets`), not these scaled
acceleration numbers. Final RMSE will be calculated in volts. Scaling does not remove
the possibility of accumulated simulation error or guarantee successful training.

We will implement these multiplications and divisions with PyTorch tensors inside the
future model so training can differentiate through them. No network is created in this section.

In [ ]:
# Conversion illustration only: 0.5 is NOT a prediction from a trained model.
example_scaled_acceleration = 0.5
example_physical_acceleration = example_scaled_acceleration * acceleration_scale
print(f"Example: scaled answer 0.5 means {example_physical_acceleration:.3f} V/s^2")

scale_summary = pd.DataFrame({
    "scale": ["y_scale", "v_scale", "u_scale", "acceleration_scale"],
    "value": [y_scale, v_scale, u_scale, acceleration_scale],
    "units": ["V", "V/s", "V", "V/s^2"],
    "source": [
        "Training output standard deviation",
        "Estimated training rate standard deviation",
        "Training input standard deviation",
        "(2*pi*training peak frequency)^2 * y_scale",
    ],
})
scale_summary

## 7. Three acceleration networks

All three models take **three numbers in physical units**, in this order:
`[output voltage y, rate v, input voltage u]`.
Each model scales them internally, predicts a scaled acceleration, and converts it back
to **acceleration in V/s²**. The future integrator will receive that physical acceleration.

| Model | Layer widths (input, hidden layers, output) | Trainable parameters |
|---|---|---:|
| A | 3 → 32 → 32 → 1 | 1,217 |
| B | 3 → 64 → 64 → 1 | 4,481 |
| C | 3 → 32 → 32 → 32 → 1 | 2,273 |

Every hidden layer uses `tanh`. The final layer is linear, with **no activation after it**.
`tanh` makes the hidden responses smooth and allows both signs. But the final linear layer
is what lets the predicted acceleration take either sign without restricting it to [-1, 1].
Even nonnegative hidden activations could produce a negative final answer with a linear output.

The models differ only in hidden layer sizes. They share the same fixed scale values,
but each has its own weights and biases. None has been trained yet.

In [ ]:
import torch
from torch import nn


class AccelerationMLP(nn.Module):
    """Physical [y, v, u] -> physical acceleration, with internal scaling."""

    def __init__(self, hidden_sizes, input_scales, acceleration_scale):
        super().__init__()
        self.hidden_sizes = tuple(hidden_sizes)

        # Buffers are saved with the model and move with it to another device.
        # They are fixed numbers, not parameters changed by the optimizer.
        self.register_buffer(
            "input_scales",
            torch.as_tensor(input_scales, dtype=torch.float32).detach().clone(),
        )
        self.register_buffer(
            "acceleration_scale",
            torch.as_tensor(acceleration_scale, dtype=torch.float32).detach().clone(),
        )

        layers = []
        previous_width = 3
        for width in self.hidden_sizes:
            layers.append(nn.Linear(previous_width, width))
            layers.append(nn.Tanh())
            previous_width = width

        layers.append(nn.Linear(previous_width, 1))  # unrestricted scalar output
        self.network = nn.Sequential(*layers)

    def forward(self, physical_inputs):
        # Last dimension must be [y, v, u]; supports one state or a batch.
        scaled_inputs = physical_inputs / self.input_scales
        scaled_acceleration = self.network(scaled_inputs)
        return scaled_acceleration * self.acceleration_scale

### How to read the class

- `nn.Linear(3, 32)` mixes the three inputs into 32 intermediate values using learned weights and biases.
- `nn.Tanh()` applies a smooth nonlinear function to those values.
- The loop adds the requested hidden layers; the final `nn.Linear(..., 1)` returns one number.
- `forward` is the calculation that happens whenever we call the model.
- `register_buffer` keeps the scales with the model without making them trainable.

The calculation is exactly

$$\alpha_\theta(y,v,u)=s_\alpha q_\theta(y/s_y,v/s_v,u/s_u).$$

All calculations in `forward` use PyTorch tensors. This keeps the path needed for training
to calculate how a weight change affects the predicted voltage through the future integrator.

**Do not give this class already-scaled inputs.** It would divide them a second time.
Use `first_network_input`, not `first_network_input_scaled`, when calling a model.

Input shape `(3,)` gives output shape `(1,)`. A batch of shape `(batch_size, 3)` gives
`(batch_size, 1)`. There is no predicted next voltage here: the output is acceleration only.

### Plain MLP versus physics-informed MLP (PINN-style)

Until now, our MLP learned only from voltage errors. RK4 alone does **not** make it a PINN:
RK4 integrates whatever acceleration the MLP produces, even if it breaks the Duffing law.

We now compare each architecture twice, with the same initial MLP weights, sampled sections,
900 updates, and learning-rate schedule:

| Family | What training asks it to do |
|---|---|
| MLP | Match measured training voltage |
| PI-MLP | Match voltage **and** make acceleration agree with the Duffing equation |

This is a **physics-informed dynamics network**, or PINN-style regularization. It keeps
our `[y,v,u] -> acceleration -> RK4` design. It is not the classical time-input PINN
`t -> y(t)` using automatic time derivatives. That time-only design would need a different
strategy for predicting the response to a future input signal. The physics-residual idea
comes from [physics-informed learning](https://maziarraissi.github.io/PINNs/).

The equation gives a second way to calculate acceleration:

$$\alpha_{physics}=gu-av-by-cy^3.$$

We do not know $a,b,c,g$, so the physics-informed model learns these four numbers too,
using **training data only**. We assume nonnegative damping and linear/cubic stiffness;
input gain can have either sign. These assumptions describe the hardening-spring model
we discussed, and should be revisited if it does not fit the circuit.

At the simulated states inside each training section, calculate

$$r=\frac{\alpha_\theta(y,v,u)-\alpha_{physics}(y,v,u)}{s_\alpha},\qquad
L=\frac{\mathrm{RMSE}_{voltage}}{s_y}+\lambda\,\mathrm{mean}(r^2).$$

In simple words: the first term says **match the recording**. The second says **follow
the equation**. We start with `lambda = 0.1`; it is a tuning choice, not a known optimum.
Both losses are dimensionless. No measured acceleration or second-difference labels are needed.
Gradients pass through both the simulated states and the two acceleration calculations.
The network still predicts acceleration at test time; we do not switch it to the fitted equation.

For stable parameter sizes, fit dimensionless $A,B,C,G$ in
$$\alpha_{physics}=s_\alpha[G(u/s_u)-A(v/s_v)-B(y/s_y)-C(y/s_y)^3].$$
Then $a=s_\alpha A/s_v$, $b=s_\alpha B/s_y$, $c=s_\alpha C/s_y^3$,
and $g=s_\alpha G/s_u$. These learned coefficients are estimates, not verified circuit constants.
The physics-informed version has four more trainable parameters than its paired MLP.

In [ ]:
class PhysicsInformedAccelerationMLP(AccelerationMLP):
    """Same acceleration MLP, plus four coefficients used in its physics loss."""

    def __init__(self, hidden_sizes, input_scales, acceleration_scale):
        super().__init__(hidden_sizes, input_scales, acceleration_scale)
        # softplus(raw) is positive. Initial values are guesses, not measured constants.
        initial_abc = torch.tensor([0.1, 1.0, 0.1], dtype=torch.float32)
        self.raw_abc = nn.Parameter(torch.log(torch.expm1(initial_abc)))
        self.gain = nn.Parameter(torch.tensor(1.0))

    def physics_acceleration(self, physical_inputs):
        ys, vs, us = (physical_inputs / self.input_scales).unbind(dim=-1)
        A, B, C = torch.nn.functional.softplus(self.raw_abc).unbind()
        return self.acceleration_scale * (self.gain * us - A * vs - B * ys - C * ys**3)

    def normalized_physics_residual(self, physical_inputs):
        return (self(physical_inputs).squeeze(-1) - self.physics_acceleration(physical_inputs)) / self.acceleration_scale

    @torch.no_grad()
    def physical_coefficients(self):
        A, B, C = torch.nn.functional.softplus(self.raw_abc).unbind()
        sy, sv, su = self.input_scales.unbind()
        sa = self.acceleration_scale
        return {"a": (sa*A/sv).item(), "b": (sa*B/sy).item(),
                "c": (sa*C/sy**3).item(), "g": (sa*self.gain/su).item()}


def build_trial_model(trial, scales, accel_scale):
    family = trial["family"]
    if family not in {"MLP", "PI-MLP"}:
        raise ValueError(f"Unknown model family: {family}")
    cls = AccelerationMLP if family == "MLP" else PhysicsInformedAccelerationMLP
    return cls(trial["hidden_sizes"], scales, accel_scale)

In [ ]:
architecture_configs = {
    "A_32_32": (32, 32),
    "B_64_64": (64, 64),
    "C_32_32_32": (32, 32, 32),
}

initialization_seed = 42
models = {}
for name, hidden_sizes in architecture_configs.items():
    # Reset the seed for reproducible initialization of each architecture.
    # Different shapes still give different models, not identical weights.
    torch.manual_seed(initialization_seed)
    models[name] = AccelerationMLP(
        hidden_sizes=hidden_sizes,
        input_scales=input_scales,
        acceleration_scale=acceleration_scale,
    )

# Each dictionary entry is a separate, untrained model.
model_a = models["A_32_32"]
model_b = models["B_64_64"]
model_c = models["C_32_32_32"]

In [ ]:
architecture_summary = pd.DataFrame([
    {
        "model": name,
        "hidden sizes": model.hidden_sizes,
        "hidden activation": "tanh",
        "output activation": "none (linear)",
        "trainable parameters": sum(
            parameter.numel() for parameter in model.parameters()
            if parameter.requires_grad
        ),
    }
    for name, model in models.items()
])
architecture_summary

### What a model call will look like

For a single initial state, the future call can be written as:

```python
physical_input = torch.as_tensor(first_network_input, dtype=torch.float32)
acceleration = model_a(physical_input)
```

This is a usage example, not a training step. Before training, its result comes from random
initial weights and is not a reliable acceleration estimate. During a simulated sequence,
the voltage and rate in the input will come from the simulated state. We will construct
those inputs with tensor operations that preserve the training calculation.

Running the model-construction cell again creates new models and discards any weights
previously learned by these variables. For now they are untrained.

## 8. RK4: move the state forward using acceleration

The state is $s=[y,v]$: output voltage and its rate of change. The network predicts
acceleration. The integrator uses that prediction to calculate the next state.

$$F(s,u)=[v,\alpha_\theta(y,v,u)].$$

This pair contains **rates**, not the next state. Multiplying rates by a small time interval
gives approximate changes. Adding those changes to the current state moves us forward.
RK4 estimates the rates four times during one step and combines them.

### Worked example: one complete step

Use the made-up starting values

$$y=0.10\ \mathrm{V},\quad v=2\ \mathrm{V/s},\quad h=0.01\ \mathrm{s}.$$

Here $h$ means the time step. The real notebook uses `dt = 1 / 610.35`, not 0.01 s.
Imagine a network that returns the physical rule

$$\alpha(y,v,u)=g u-\omega^2 y,$$

with $g=1\ \mathrm{s}^{-2}$ and $\omega^2=100\ \mathrm{s}^{-2}$.
Keep $u=0$ for this entire example, so numerically the acceleration is $-100y$.
This is a toy rule for arithmetic, not the output of one of our untrained MLPs.

**1. Rates at the starting state**

$$\alpha_1=-100(0.10)=-10,\qquad k_1=[2,-10].$$

**2. First halfway estimate**

Half a step is 0.005 s. Start from the original state and use $k_1$:

$$s_{\mathrm{half},1}=[0.10,2]+0.005[2,-10]=[0.11,1.95].$$

The network gives $\alpha_2=-100(0.11)=-11$, so

$$k_2=[1.95,-11].$$

**3. Improved halfway estimate**

Start from the **original state again**, now using $k_2$:

$$s_{\mathrm{half},2}=[0.10,2]+0.005[1.95,-11]=[0.10975,1.945].$$

The network gives $\alpha_3=-100(0.10975)=-10.975$, so

$$k_3=[1.945,-10.975].$$

**4. Estimated end state**

Start from the original state and use $k_3$ for a full step:

$$s_{\mathrm{end}}=[0.10,2]+0.01[1.945,-10.975]=[0.11945,1.89025].$$

The network gives $\alpha_4=-100(0.11945)=-11.945$, so

$$k_4=[1.89025,-11.945].$$

**Combine the four estimates**

The two halfway estimates get twice the weight:

$$s_{\mathrm{next}}=s+\frac{h}{6}(k_1+2k_2+2k_3+k_4).$$

For voltage:

$$y_{\mathrm{next}}=0.10+\frac{0.01}{6}
[2+2(1.95)+2(1.945)+1.89025]\approx\boxed{0.119467\ \mathrm{V}}.$$

For its rate:

$$v_{\mathrm{next}}=2+\frac{0.01}{6}
[-10+2(-11)+2(-10.975)-11.945]=\boxed{1.890175\ \mathrm{V/s}}.$$

The intermediate states were temporary estimates. Only this final pair becomes the next
simulation state. All four calls use the same network weights and fixed scales.

### What input voltage does RK4 use halfway between samples?

In the example, input stayed at zero. In our recording it changes, and we only know its
values at the sample times. We choose **linear interpolation** between neighboring inputs:

$$u_{\mathrm{half}}=(u_k+u_{k+1})/2.$$

The four calls use $u_k$, $u_{\mathrm{half}}$, $u_{\mathrm{half}}$, and $u_{k+1}$.
This assumes the input follows a straight line between samples. It is a modeling choice,
not an extra measurement. Reading the next input is allowed here because the applied input
record is known, including during testing. No next measured output voltage is read.

In the functions below, `state` is a PyTorch tensor containing `[y, v]` in physical units.
The model already handles scaling, so the integrator must not scale these values again.

RK4 approximates the differential equation with our chosen interpolated input. It does
not remove errors in the learned acceleration, initial rate, or assumed between-sample
input. A long free run is still needed to assess the resulting predictions.

In [ ]:
def state_derivative(model, state, input_voltage):
    """Return [dy/dt, dv/dt] for a physical state of shape (..., 2).

    input_voltage is a scalar or has the state's leading batch shape.
    The state must share the model's floating-point dtype and device.
    """
    y = state[..., 0]
    v = state[..., 1]
    u = torch.as_tensor(input_voltage, dtype=state.dtype, device=state.device)
    u = torch.broadcast_to(u, y.shape)

    physical_inputs = torch.stack((y, v, u), dim=-1)
    acceleration = model(physical_inputs).squeeze(-1)
    return torch.stack((v, acceleration), dim=-1)


def rk4_step(model, state, u_start, u_end, dt):
    """Advance [y, v] by one dt using a linearly interpolated input.

    Returns a new state tensor of the same shape; does not change state in place.
    """
    u_start = torch.as_tensor(u_start, dtype=state.dtype, device=state.device)
    u_end = torch.as_tensor(u_end, dtype=state.dtype, device=state.device)
    u_half = 0.5 * (u_start + u_end)

    k1 = state_derivative(model, state, u_start)
    k2 = state_derivative(model, state + 0.5 * dt * k1, u_half)
    k3 = state_derivative(model, state + 0.5 * dt * k2, u_half)
    k4 = state_derivative(model, state + dt * k3, u_end)

    return state + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

### How this becomes a sequence

`rk4_step` moves forward once. `rk4_rollout` repeats it in time order, carrying the
calculated state into the next step. It receives an initial state and the input voltage
sequence. It does not receive measured output voltages or labels.

For N input samples, we make N - 1 steps and return N states, including the initial state.
This keeps predictions aligned with measurements: `states[0]` is at the initial time,
`states[1]` is one time step later, and so on.

For one sequence, input shape is `(N,)`, initial state shape is `(2,)`, and returned
states have shape `(N, 2)`. For future batches, inputs may have shape `(N, batch_size)`
with initial state `(batch_size, 2)`; the returned shape is `(N, batch_size, 2)`.

The functions use tensor operations without detaching states from the training calculation.
Later, PyTorch can follow how the weights affect voltage predictions through all the steps.
Defining these functions does not train a model or run a simulation.

In [ ]:
def rk4_rollout(model, initial_state, input_voltages, dt):
    """Simulate at sample times; axis 0 of input_voltages is time.

    initial_state is a tensor on the model's dtype/device, in physical units.
    Returned states include the initial state; all inputs are in physical volts.
    """
    inputs = torch.as_tensor(
        input_voltages, dtype=initial_state.dtype, device=initial_state.device
    )
    if inputs.ndim == 0 or inputs.shape[0] == 0:
        raise ValueError("Provide at least one input-voltage sample along axis 0.")

    state = initial_state
    states = [state]
    for k in range(inputs.shape[0] - 1):
        state = rk4_step(model, state, inputs[k], inputs[k + 1], dt)
        states.append(state)

    return torch.stack(states, dim=0)

### How we will call the integrator later

This is a usage example, kept as Markdown so it does not automatically simulate an untrained model:

```python
reference_parameter = next(model_a.parameters())
start_state = torch.as_tensor(
    initial_state,
    dtype=reference_parameter.dtype,
    device=reference_parameter.device,
)

# One step: returns physical [next output voltage, next rate].
next_state = rk4_step(model_a, start_state, u_train[0], u_train[1], dt)

# A whole training sequence: column 0 is voltage, column 1 is its rate.
states = rk4_rollout(model_a, start_state, u_train, dt)
predicted_training_voltage = states[:, 0]
```

The three models are still untrained, so these calls would not yet give useful predictions.
Do not substitute `u_train_scaled` for `u_train`: scaling is inside the model.

For final evaluation after training, we will pass the full input `u` in one call, keeping
the same state across the train-test boundary. We will then split the returned voltage
predictions for scoring. We will not initialize a new state from test measurements.

## 9. RMSE and training sequences: two different choices

Task 5 (pages 5-6) requires **RMSE from one free run of the entire excerpt** for the final
score. It also asks us to tune the length of the rollout used for training. These fit together:

- **RMSE** tells us how to measure error.
- **Training rollout length** tells us how far to simulate before calculating a training error
  and updating the weights.

For predicted voltages $\hat y$ and measured voltages $y$,

$$\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2}.$$

We will train with RMSE too. For the optimizer, we divide it by the fixed training voltage
scale, `y_scale`, to make the loss dimensionless. This is just multiplication by a constant;
it does not change which predictions minimize the error. Every displayed RMSE stays in volts.

### What does choosing a sequence mean?

Suppose we choose training index 100 and a rollout length of 32 **steps**:

1. Initialize from `y_train[100]` and `v_train_est[100]`.
2. Read input samples 100 through 132, in their original order.
3. Make 32 RK4 steps, carrying the simulated voltage and rate forward.
4. Compare the 32 predicted future voltages with `y_train[101:133]`.

The initial voltage is provided, so we exclude it from this training loss. Measured voltages
inside the section are labels, not replacements for the simulated state.

A **batch** contains several such sections. Their starting indices can be random, but the
samples inside every section stay in time order. All sections are fully inside the first
3072 samples; initialization rates were estimated from those training samples only.

We start with 32-step sections, then increase to 64 and 128 steps. These are initial settings
to try, not known best settings. Short sections make the first learning problem easier.
Longer ones expose more accumulated error.

**Only training uses these separate starts.** Final evaluation starts once at excerpt sample 0
and continues through all 4096 samples, without resetting at the test boundary. A low training
section RMSE does not guarantee a low final free-run RMSE.

In [ ]:
import math
import time


def voltage_rmse(predicted, measured):
    """RMSE over matching, nonempty PyTorch tensors; inputs are in volts."""
    if predicted.shape != measured.shape or predicted.numel() == 0:
        raise ValueError("RMSE needs nonempty prediction and target tensors of the same shape.")
    error = predicted - measured
    # Equivalent to sqrt(mean(error**2)); the norm handles zero error in backpropagation.
    return torch.linalg.vector_norm(error) / math.sqrt(error.numel())


def sample_training_sections(u_values, y_values, v_estimates, rollout_steps,
                             batch_size, generator):
    """Draw ordered sections from training-only 1D tensors.

    L steps require L+1 samples. Return time-first tensors for rk4_rollout.
    """
    sample_count = len(y_values)
    if not 1 <= rollout_steps < sample_count:
        raise ValueError("Rollout steps must be between 1 and the training sample count minus 1.")
    if batch_size < 1:
        raise ValueError("Batch size must be positive.")

    # Largest possible start is sample_count - rollout_steps - 1.
    starts = torch.randint(
        low=0, high=sample_count - rollout_steps,
        size=(batch_size,), generator=generator,
    ).to(y_values.device)
    offsets = torch.arange(rollout_steps + 1, device=y_values.device)
    indices = offsets[:, None] + starts[None, :]

    starting_states = torch.stack((y_values[starts], v_estimates[starts]), dim=-1)
    section_inputs = u_values[indices]       # (L+1, batch)
    section_targets = y_values[indices]     # (L+1, batch)
    return starting_states, section_inputs, section_targets

### One weight update

The loop simulates a batch of training sections, compares voltage predictions with labels,
and differentiates the loss through RK4. For PI-MLP it also calculates the Duffing residual
at those simulated training states and fits the four equation coefficients. Gradient
clipping limits large updates. Reported data and physics losses describe the state before
each update; checkpoint weights are saved after the successful update.

No test targets or test states enter this function. Periodic checkpoints contain enough
state to restart from the last saved update, provided the files have been retained.

### Checkpoints: saving a file and keeping it are two separate steps

The previous notebook saved only after each whole trial. This version saves **before
training, every 50 successful updates, at every stage end, and at completion**.
Each checkpoint contains weights, fixed scales, learned physics coefficients if present,
optimizer state, completed-update count, learning history, section-sampling RNG state,
PyTorch RNG states, configuration, and preprocessing information.

The code writes a temporary checkpoint, loads it back, checks every model tensor, then
replaces `TRIAL_latest.pt`. It keeps `TRIAL_previous.pt` as a fallback. A completed
model also gets `TRIAL_final.pt`. `checkpoint_inventory.csv` records sizes and SHA-256 hashes.
This follows [PyTorch's general checkpoint approach](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

Every checkpoint also refreshes a ZIP **outside** the run folder, so the ZIP never includes
itself. Download links appear before training, after each completed model, and after plots.
The first ZIP contains only initialization; download a newer ZIP after training progresses.

**Files in `/kaggle/working` are still session files. Python cannot guarantee that Kaggle
will retain a discarded session.** Use **Save Version -> Save & Run All** for the new run
so that a successfully completed version retains its outputs. This starts a fresh run;
do not also launch the same training interactively. For an interactive run, download the
ZIP while the session is alive. A downloaded `.ipynb` alone is not a weights backup.
See [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks).

To resume, extract your downloaded ZIP, attach the extracted run folder as a Kaggle
dataset, and set `RESUME_RUN_DIR` below to that folder (the one containing `run_metadata.json`).
You can also use an existing local run folder. The code copies checkpoints into a **new
writable run**, restores unfinished trials, and skips already completed ones.
Keep the experiment settings unchanged when resuming. Old weights-only checkpoints do
not contain enough information for this resume path.

An interrupted trial loses at most the work since its last saved checkpoint **if those
files survive**. An abrupt session deletion can still remove every unexported file.
Seeds and saved RNG states improve repeatability; different PyTorch/CUDA versions or
devices can still change numerical results. This is not a promise of bitwise reproducibility.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import zipfile
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import FileLink, display

CHECKPOINT_FORMAT = 2


def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return value


def atomic_json(value, destination):
    destination = Path(destination)
    temporary = destination.with_name(destination.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    os.replace(temporary, destination)


def load_checkpoint(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if checkpoint.get("format_version") != CHECKPOINT_FORMAT:
        raise ValueError("Unsupported checkpoint format; use checkpoints from this notebook version.")
    required = {"model_state_dict", "optimizer_state_dict", "update", "history",
                "sampler_rng_state", "torch_rng_state", "cuda_rng_states", "trial", "preprocessing"}
    if not required.issubset(checkpoint):
        raise ValueError(f"Incomplete checkpoint: {checkpoint_path}")
    total = sum(stage["updates"] for stage in checkpoint["trial"]["schedule"])
    if not 0 <= checkpoint["update"] <= total or len(checkpoint["history"]) != checkpoint["update"]:
        raise ValueError("Checkpoint update count and history disagree.")
    if bool(checkpoint["complete"]) != (checkpoint["update"] == total):
        raise ValueError("Checkpoint completion flag is inconsistent.")
    return checkpoint


def write_verified_checkpoint(checkpoint, destination):
    destination = Path(destination)
    temporary = destination.with_name(destination.name + ".tmp")
    # Flush Python and OS buffers before checking the file we actually wrote.
    with temporary.open("wb") as stream:
        torch.save(checkpoint, stream)
        stream.flush()
        os.fsync(stream.fileno())
    restored = load_checkpoint(temporary)
    expected = checkpoint["model_state_dict"]
    actual = restored["model_state_dict"]
    if expected.keys() != actual.keys() or any(not torch.equal(expected[k], actual[k]) for k in expected):
        raise IOError("Checkpoint read-back failed; the previous checkpoint was not replaced.")
    if destination.exists():
        previous = destination.with_name(destination.name.replace("_latest.pt", "_previous.pt"))
        previous_temp = previous.with_name(previous.name + ".tmp")
        shutil.copy2(destination, previous_temp)
        os.replace(previous_temp, previous)
    os.replace(temporary, destination)


def refresh_download_bundle(folder, show_link=False):
    folder = Path(folder).resolve()
    inventory = []
    for checkpoint_path in sorted(folder.glob("*.pt")):
        inventory.append({"file": checkpoint_path.name, "bytes": checkpoint_path.stat().st_size,
                          "sha256": hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()})
    pd.DataFrame(inventory, columns=["file", "bytes", "sha256"]).to_csv(
        folder / "checkpoint_inventory.csv", index=False)
    archive = folder.parent.parent / f"silverbox_{folder.name}.zip"
    temporary = archive.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for item in sorted(folder.rglob("*")):
            if item.is_file() and not item.name.endswith(".tmp"):
                bundle.write(item, arcname=str(Path(folder.name) / item.relative_to(folder)))
    with zipfile.ZipFile(temporary) as bundle:
        failed_file = bundle.testzip()
        if failed_file is not None:
            raise IOError(f"ZIP verification failed: {failed_file}")
    os.replace(temporary, archive)
    if show_link:
        print(f"DOWNLOAD THIS ZIP to keep the files: {archive} ({archive.stat().st_size:,} bytes)")
        display(FileLink(os.path.relpath(archive, Path.cwd())))
        print("If the link is unavailable in Kaggle, download this ZIP from its Output/files panel.")
    return archive


def make_checkpoint_callback(folder, trial, preprocessing):
    def save(model, optimizer, generator, update, history, complete):
        checkpoint = {
            "format_version": CHECKPOINT_FORMAT,
            "model_state_dict": cpu_tree(model.state_dict()),
            "optimizer_state_dict": cpu_tree(optimizer.state_dict()),
            "sampler_rng_state": generator.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "update": update, "complete": complete, "history": list(history),
            "trial": trial, "preprocessing": preprocessing,
            "dtype": "torch.float32", "torch_version": str(torch.__version__),
        }
        destination = folder / f"{trial['name']}_latest.pt"
        write_verified_checkpoint(checkpoint, destination)
        if complete:
            final_temp = folder / f"{trial['name']}_final.pt.tmp"
            shutil.copy2(destination, final_temp)
            os.replace(final_temp, folder / f"{trial['name']}_final.pt")
        pd.DataFrame(history).to_csv(folder / f"{trial['name']}_history.csv", index=False)
        refresh_download_bundle(folder, show_link=(update == 0 or complete))
        print(f"Checkpoint verified: {trial['name']}, {update} updates, {destination.name}")
    return save


def restore_trial_for_prediction(checkpoint_path, device="cpu"):
    """Rebuild a trained model from disk without retraining; definitions above must be run."""
    checkpoint = load_checkpoint(checkpoint_path)
    prep = checkpoint["preprocessing"]
    model = build_trial_model(checkpoint["trial"], prep["input_scales"], prep["acceleration_scale"])
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.to(device=device).eval(), checkpoint

In [ ]:
def train_acceleration_model(model, u_training, y_training, v_training_est, dt,
                             schedule, batch_size=32, optimizer_name="Adam",
                             seed=42, gradient_clip=1.0, log_every=50,
                             physics_weight=0.0, checkpoint_every=50,
                             checkpoint_callback=None, resume=None):
    """Train only on the supplied training arrays; optionally resume a complete optimizer state."""
    reference = next(model.parameters())
    options = {"dtype": reference.dtype, "device": reference.device}
    u_values, y_values, v_values = [torch.as_tensor(a, **options)
                                  for a in (u_training, y_training, v_training_est)]
    if any(a.ndim != 1 or a.shape != y_values.shape for a in (u_values, y_values, v_values)):
        raise ValueError("Training arrays must be matching 1D arrays.")
    if not all(torch.isfinite(a).all().item() for a in (u_values, y_values, v_values)):
        raise ValueError("Training arrays must be finite.")
    if min(batch_size, log_every, checkpoint_every, gradient_clip, dt) <= 0 or physics_weight < 0:
        raise ValueError("Intervals, batch size, dt and gradient limit must be positive; physics weight nonnegative.")
    if physics_weight and not isinstance(model, PhysicsInformedAccelerationMLP):
        raise ValueError("A physics loss requires the physics-informed model.")
    if not schedule or any(not 1 <= s["rollout_steps"] < len(y_values)
                           or s["updates"] < 1 or s["learning_rate"] <= 0 for s in schedule):
        raise ValueError("Provide positive updates/rates and rollout lengths within training data.")
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=schedule[0]["learning_rate"])
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=schedule[0]["learning_rate"], weight_decay=1e-4)
    else:
        raise ValueError("Choose Adam or AdamW.")

    generator = torch.Generator().manual_seed(seed)
    history, update_number = [], 0
    if resume is not None:
        model.load_state_dict(resume["model_state_dict"], strict=True)
        optimizer.load_state_dict(resume["optimizer_state_dict"])
        generator.set_state(resume["sampler_rng_state"])
        torch.set_rng_state(resume["torch_rng_state"])
        if model.input_scales.is_cuda and resume["cuda_rng_states"]:
            if len(resume["cuda_rng_states"]) == torch.cuda.device_count():
                torch.cuda.set_rng_state_all(resume["cuda_rng_states"])
            else:
                print("CUDA device count changed; exact RNG continuation is unavailable.")
        history = list(resume["history"])
        update_number = int(resume["update"])
        print(f"Restored model, optimizer and sampling state after {update_number} updates.")
    model.train()
    total_updates = sum(s["updates"] for s in schedule)
    prior_elapsed = history[-1]["elapsed_seconds"] if history else 0.0
    started_at = time.perf_counter()
    if checkpoint_callback is not None:
        checkpoint_callback(model, optimizer, generator, update_number, history, update_number == total_updates)

    preceding_updates = 0
    for stage_number, stage in enumerate(schedule, start=1):
        stage_end = preceding_updates + stage["updates"]
        if update_number >= stage_end:
            preceding_updates = stage_end
            continue
        for group in optimizer.param_groups:
            group["lr"] = stage["learning_rate"]
        print(f"Stage {stage_number}: {stage['rollout_steps']} steps; lr={stage['learning_rate']:g}")
        while update_number < stage_end:
            starts, section_inputs, targets = sample_training_sections(
                u_values, y_values, v_values, stage["rollout_steps"], batch_size, generator)
            optimizer.zero_grad(set_to_none=True)
            states = rk4_rollout(model, starts, section_inputs, dt)
            section_rmse = voltage_rmse(states[1:, :, 0], targets[1:])
            data_loss = section_rmse / model.input_scales[0]
            physics_mse = data_loss.new_zeros(())
            if physics_weight > 0:
                # Only states and inputs inside these TRAINING sections; no test collocation points.
                physics_inputs = torch.cat((states, section_inputs.unsqueeze(-1)), dim=-1)
                physics_mse = model.normalized_physics_residual(physics_inputs).square().mean()
            loss = data_loss + physics_weight * physics_mse
            if not torch.isfinite(loss).item():
                raise FloatingPointError("Non-finite training loss. Last verified checkpoint remains available.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip, error_if_nonfinite=True)
            optimizer.step()
            if not all(torch.isfinite(p).all().item() for p in model.parameters()):
                raise FloatingPointError("Non-finite updated weights; last verified checkpoint was kept.")
            update_number += 1
            history.append({
                "update": update_number, "stage": stage_number,
                "rollout_steps": stage["rollout_steps"], "learning_rate": stage["learning_rate"],
                "section_rmse_V": section_rmse.detach().item(),
                "normalized_data_loss": data_loss.detach().item(),
                "physics_mse_normalized": physics_mse.detach().item(),
                "weighted_physics_loss": (physics_weight * physics_mse).detach().item(),
                "normalized_loss": loss.detach().item(),
                "elapsed_seconds": prior_elapsed + time.perf_counter() - started_at,
            })
            if update_number % log_every == 0 or update_number == stage_end:
                print(f"  Update {update_number}: section RMSE={history[-1]['section_rmse_V']:.6f} V; "
                      f"physics MSE={history[-1]['physics_mse_normalized']:.5g}")
            if checkpoint_callback is not None and (update_number % checkpoint_every == 0 or update_number == stage_end):
                checkpoint_callback(model, optimizer, generator, update_number, history, update_number == total_updates)
        preceding_updates = stage_end
    return pd.DataFrame(history)

## 10. The assignment's final free-run metric

After a model's weights are fixed, simulate the full input `u` in **one** call to `rk4_rollout`.
The initial state is `[y0, v0]`, obtained at the start of training. Only after simulation
do we compare the predicted voltage with the measured output:

$$\mathrm{RMSE}_{\mathrm{train}}=
\sqrt{\frac{1}{3072}\sum_{k=0}^{3071}(\hat y_k-y_k)^2},$$
$$\mathrm{RMSE}_{\mathrm{test}}=
\sqrt{\frac{1}{1024}\sum_{k=3072}^{4095}(\hat y_k-y_k)^2}.$$

The split is used only for scoring. The simulation does not reset there.
The training score includes its first sample, as required, even though that voltage was
supplied as the initial condition. The zero-prediction baseline is computed from the actual
test targets, rather than assuming it is exactly 0.06 V.

`torch.no_grad()` disables gradient recording for this evaluation; weights are not updated.
A non-finite trajectory is reported as a failed free run, never a good score.

In [ ]:
@torch.no_grad()
def evaluate_free_run(model, input_voltages, measured_output, starting_state, dt,
                      split_index):
    """One continuous simulation, followed by training/test voltage RMSE."""
    reference = next(model.parameters())
    tensor_options = {"dtype": reference.dtype, "device": reference.device}
    inputs = torch.as_tensor(input_voltages, **tensor_options)
    start = torch.as_tensor(starting_state, **tensor_options)
    if inputs.ndim != 1 or start.shape != (2,) or not 0 < split_index < len(inputs):
        raise ValueError("Evaluation needs one input sequence, a [y0, v0] state, and an interior split.")

    was_training = model.training
    model.eval()
    try:
        states = rk4_rollout(model, start, inputs, dt)
    finally:
        model.train(was_training)
    prediction = states[:, 0]

    # Measured output enters only AFTER simulation, for scoring.
    target = torch.as_tensor(measured_output, **tensor_options)
    if target.shape != prediction.shape or not torch.isfinite(target).all().item():
        raise ValueError("Evaluation targets must be finite and match the simulated voltage shape.")
    baseline = voltage_rmse(torch.zeros_like(target[split_index:]), target[split_index:])
    finite_trajectory = torch.isfinite(states).all().item()
    if finite_trajectory:
        train_score = voltage_rmse(prediction[:split_index], target[:split_index]).item()
        test_score = voltage_rmse(prediction[split_index:], target[split_index:]).item()
    else:
        train_score = float("inf")
        test_score = float("inf")

    return {
        "train_free_run_rmse_V": train_score,
        "test_free_run_rmse_V": test_score,
        "zero_test_baseline_V": baseline.item(),
        "finite_trajectory": finite_trajectory,
        "predicted_voltage": prediction.cpu(),
    }

## 11. Paired experiment: three MLPs and three physics-informed MLPs

There are **six trials**, each with 900 updates: 300 at 32 steps, 300 at 64, and 300 at 128.
An update is one batch of 32 sections, not an epoch. Each architecture's MLP/PI-MLP pair
gets the same initial network weights, seed, sampled sections, and optimizer schedule.
The PI-MLP adds four fitted coefficients and a physics penalty, so its update costs more.
This experiment will take longer than the previous three-model run.

The comparison reports both families and their differences. One seed and one physics weight
are a starting comparison, not enough to conclude that a family is always better.
Test measurements never enter either training objective. Test RMSE selects the best trial
only because the assignment explicitly requests that rule; it is not an untouched validation study.

`RESUME_RUN_DIR=None` starts a new experiment. Set it to a saved run folder to continue.
The next execution cell starts training; setup and visualization definitions alone do not.

In [ ]:
device_choice = "auto"  # "auto", "cpu", or "cuda"
RESUME_RUN_DIR = None  # Example: "/kaggle/input/my-checkpoints/20261011T..."
CHECKPOINT_EVERY = 50
if device_choice not in {"auto", "cpu", "cuda"}:
    raise ValueError("Choose auto, cpu, or cuda.")
if device_choice == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a Kaggle GPU or choose cpu.")
training_device = torch.device("cuda" if device_choice == "auto" and torch.cuda.is_available()
                               else "cpu" if device_choice == "auto" else device_choice)
training_dtype = torch.float32
print(f"PyTorch {torch.__version__}; training device: {training_device}")
if training_device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(training_device)}")

initial_schedule = [
    {"rollout_steps": 32, "updates": 300, "learning_rate": 1e-3},
    {"rollout_steps": 64, "updates": 300, "learning_rate": 3e-4},
    {"rollout_steps": 128, "updates": 300, "learning_rate": 1e-4},
]
experiment_plan = [
    {"name": f"{architecture}_{family.replace('-', '_')}_adam",
     "architecture": architecture, "hidden_sizes": list(widths), "family": family,
     "physics_weight": weight, "optimizer": "Adam", "batch_size": 32,
     "seed": 42, "gradient_clip": 1.0,
     "schedule": [dict(stage) for stage in initial_schedule]}
    for architecture, widths in architecture_configs.items()
    for family, weight in [("MLP", 0.0), ("PI-MLP", 0.1)]
]
pd.DataFrame([{k: v for k, v in trial.items() if k != "schedule"} for trial in experiment_plan])

In [ ]:
# TRAINING STARTS HERE. Completed trials save/download separately; this cell can be resumed.
import copy
import re

if len({trial["name"] for trial in experiment_plan}) != len(experiment_plan):
    raise ValueError("Every trial must have a distinct name.")
if any(re.fullmatch(r"[A-Za-z0-9_-]+", trial["name"]) is None for trial in experiment_plan):
    raise ValueError("Trial names may contain only letters, digits, underscores and hyphens.")

preprocessing_metadata = {
    "csv_filename": path.name,
    "excerpt_sha256": hashlib.sha256(np.column_stack((u_raw, y_raw)).astype("<f8").tobytes()).hexdigest(),
    "excerpt_start": excerpt_start, "excerpt_stop": excerpt_stop,
    "training_sample_count": training_sample_count, "sampling_frequency": sampling_frequency,
    "dt": dt, "centering_scope": "full excerpt as required by assignment",
    "u_mean": float(u_mean), "y_mean": float(y_mean), "input_order": ["y", "v", "u"],
    "input_scales": input_scales.tolist(), "acceleration_scale": float(acceleration_scale),
    "dominant_frequency_hz": float(dominant_frequency_hz), "initial_state": initial_state.tolist(),
    "initial_rate_method": "forward difference of first two training outputs",
    "input_interpolation": "linear",
}

resume_folder = Path(RESUME_RUN_DIR).expanduser() if RESUME_RUN_DIR is not None else None
resume_checkpoints = {}
if resume_folder is not None:
    old_metadata = json.loads((resume_folder / "run_metadata.json").read_text(encoding="utf-8"))
    if old_metadata["preprocessing"] != preprocessing_metadata or old_metadata["experiment_plan"] != experiment_plan:
        raise ValueError("Resume settings or data differ. Restore the original settings before continuing.")
    for trial in experiment_plan:
        candidates = [resume_folder / f"{trial['name']}_{suffix}.pt" for suffix in ("final", "latest", "previous")]
        existing = [p for p in candidates if p.is_file()]
        if not existing:
            continue  # Trial had not started.
        loaded = None
        for checkpoint_path in existing:
            try:
                loaded = load_checkpoint(checkpoint_path)
                break
            except Exception as error:
                print(f"Cannot read {checkpoint_path.name}: {error}; trying the next saved copy.")
        if loaded is None:
            raise ValueError(f"No readable checkpoint for {trial['name']}; refusing to silently restart it.")
        if loaded["trial"] != trial or loaded["preprocessing"] != preprocessing_metadata:
            raise ValueError(f"Checkpoint configuration mismatch for {trial['name']}.")
        resume_checkpoints[trial["name"]] = loaded

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
run_dir = output_root / run_id
run_dir.mkdir(parents=True, exist_ok=False)
run_metadata = {
    "checkpoint_format": CHECKPOINT_FORMAT, "run_id": run_id,
    "resumed_from": str(resume_folder) if resume_folder is not None else None,
    "python": platform.python_version(), "torch": str(torch.__version__),
    "numpy": str(np.__version__), "pandas": str(pd.__version__),
    "device": str(training_device), "dtype": str(training_dtype),
    "gpu": torch.cuda.get_device_name(training_device) if training_device.type == "cuda" else None,
    "preprocessing": preprocessing_metadata, "experiment_plan": experiment_plan,
    "checkpoint_every": CHECKPOINT_EVERY, "AdamW_weight_decay": 1e-4,
}
atomic_json(run_metadata, run_dir / "run_metadata.json")
(run_dir / "README.txt").write_text(
    "This folder contains weights and optimizer checkpoints, not just notebook logs.\n"
    "Resume: run notebook setup, set RESUME_RUN_DIR to this folder, then run the training cell.\n"
    "Keep the experiment settings unchanged. Completed trials are skipped.\n"
    "latest.pt is the most recent verified checkpoint; previous.pt is its predecessor.\n"
    "final.pt means all configured updates completed, not best validation score.\n"
    "Download the ZIP or retain Kaggle version outputs before ending a session.\n",
    encoding="utf-8")

# Preserve all recovered trials immediately, even if a later training trial fails.
for name, checkpoint in resume_checkpoints.items():
    write_verified_checkpoint(checkpoint, run_dir / f"{name}_latest.pt")
    if checkpoint["complete"]:
        shutil.copy2(run_dir / f"{name}_latest.pt", run_dir / f"{name}_final.pt")
    pd.DataFrame(checkpoint["history"]).to_csv(run_dir / f"{name}_history.csv", index=False)
print(f"Run folder: {run_dir.resolve()}")
refresh_download_bundle(run_dir, show_link=True)

trained_models, training_histories, completed_trials = {}, {}, {}
failed_trials = {}
for original_trial in experiment_plan:
    trial = copy.deepcopy(original_trial)
    name = trial["name"]
    torch.manual_seed(trial["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(trial["seed"])
    candidate = build_trial_model(trial, input_scales, acceleration_scale).to(
        device=training_device, dtype=training_dtype)
    restored = resume_checkpoints.get(name)
    if restored is not None and restored["complete"]:
        candidate.load_state_dict(restored["model_state_dict"], strict=True)
        history = pd.DataFrame(restored["history"])
        print(f"Loaded completed {name}; no retraining.")
    else:
        print(f"\nTraining {name}")
        try:
            history = train_acceleration_model(
                candidate, u_train, y_train_targets, v_train_est, dt,
                schedule=trial["schedule"], batch_size=trial["batch_size"],
                optimizer_name=trial["optimizer"], seed=trial["seed"],
                gradient_clip=trial["gradient_clip"], physics_weight=trial["physics_weight"],
                checkpoint_every=CHECKPOINT_EVERY,
                checkpoint_callback=make_checkpoint_callback(run_dir, trial, preprocessing_metadata),
                resume=restored)
        except FloatingPointError as error:
            failed_trials[name] = str(error)
            atomic_json(failed_trials, run_dir / "failed_trials.json")
            refresh_download_bundle(run_dir, show_link=True)
            print(f"FAILED {name}: {error}. Continuing with the other trials.")
            continue
    candidate.eval()
    trained_models[name], training_histories[name], completed_trials[name] = candidate, history, trial
    refresh_download_bundle(run_dir, show_link=True)
print(f"Completed {len(trained_models)}/{len(experiment_plan)} trials. Save/download the latest ZIP.")

### Compare completed trials using the required score

Run the following cell after training. It scores each completed model once on the same
whole-excerpt free run. The best version is the one with the smallest test RMSE, following
the assignment's explicit selection rule. Test output never participates in weight updates.

`training_histories` contains section errors during optimization. `comparison` contains
the required training and test free-run errors. These are different measurements.

In [ ]:
if not trained_models:
    raise ValueError("No completed models to evaluate. Download the checkpoints and inspect training failures.")

evaluation_results, prediction_tables, comparison_rows = {}, {}, []
rolling_window = 64  # trailing ~105 ms; reset the window at the scoring boundary
for name, model in trained_models.items():
    result = evaluate_free_run(model, u, y, initial_state, dt, training_sample_count)
    evaluation_results[name] = result
    prediction = result["predicted_voltage"].numpy().astype(float)
    residual = prediction - y  # positive means predicted voltage is too high
    finite_error = np.where(np.isfinite(residual), residual, np.nan)
    local_rmse = np.full(len(y), np.nan)
    for section in (slice(0, training_sample_count), slice(training_sample_count, len(y))):
        # Require a whole finite window: failed simulations cannot look good by dropping NaNs.
        local_rmse[section] = np.sqrt(pd.Series(finite_error[section]**2).rolling(
            rolling_window, min_periods=rolling_window).mean().to_numpy())
    table = pd.DataFrame({
        "trial": name, "family": completed_trials[name]["family"],
        "sample_index": np.arange(len(y)), "source_csv_row": excerpt_start + np.arange(len(y)),
        "time_s": t, "input_voltage_V": u, "measured_output_V": y,
        "predicted_output_V": prediction, "residual_V": residual,
        "absolute_error_V": np.abs(residual), "squared_error_V2": residual**2,
        "rolling_64_sample_rmse_V": local_rmse,
        "segment": np.where(np.arange(len(y)) < training_sample_count, "train", "test"),
    })
    prediction_tables[name] = table
    table.to_csv(run_dir / f"{name}_free_run.csv", index=False)
    trial = completed_trials[name]
    row = {"trial": name, "family": trial["family"], "architecture": trial["architecture"],
           "hidden_sizes": trial["hidden_sizes"], "physics_weight": trial["physics_weight"],
           "trainable_parameters": sum(p.numel() for p in model.parameters()),
           "optimizer": trial["optimizer"], "seed": trial["seed"], "batch_size": trial["batch_size"],
           "updates": len(training_histories[name]),
           "training_seconds": float(training_histories[name]["elapsed_seconds"].iloc[-1]),
           **{k: v for k, v in result.items() if k != "predicted_voltage"}}
    row["beats_zero_baseline"] = result["test_free_run_rmse_V"] < result["zero_test_baseline_V"]
    row["test_max_absolute_error_V"] = float(np.max(np.abs(residual[training_sample_count:]))) if result["finite_trajectory"] else float("inf")
    if isinstance(model, PhysicsInformedAccelerationMLP):
        row.update({f"learned_{k}": v for k, v in model.physical_coefficients().items()})
    comparison_rows.append(row)

comparison = pd.DataFrame(comparison_rows).sort_values("test_free_run_rmse_V").reset_index(drop=True)
comparison.to_csv(run_dir / "comparison.csv", index=False)
pd.concat(prediction_tables.values(), ignore_index=True).to_csv(run_dir / "all_models_residuals.csv", index=False)
valid = comparison[comparison["finite_trajectory"] & np.isfinite(comparison["test_free_run_rmse_V"])]
best_name = None if valid.empty else valid.iloc[0]["trial"]
best_model = trained_models.get(best_name)
if best_name is not None:
    (run_dir / "best_trial.txt").write_text(best_name, encoding="utf-8")
    print(f"Lowest test RMSE: {best_name} = {evaluation_results[best_name]['test_free_run_rmse_V']:.6f} V")
display(comparison)

# Match on architecture; retain missing/failed partners explicitly instead of dropping them.
paired_comparison = comparison.pivot(index="architecture", columns="family", values="test_free_run_rmse_V")
paired_comparison = paired_comparison.reindex(index=list(architecture_configs), columns=["MLP", "PI-MLP"])
paired_comparison["PI_minus_MLP_V"] = paired_comparison["PI-MLP"] - paired_comparison["MLP"]
paired_comparison["PI_improvement_percent"] = 100 * (paired_comparison["MLP"] - paired_comparison["PI-MLP"]) / paired_comparison["MLP"].replace(0, np.nan)
paired_comparison.to_csv(run_dir / "paired_comparison.csv")
print("Negative PI_minus_MLP_V means the physics-informed version did better. NaN means a missing/undefined comparison.")
display(paired_comparison)
if failed_trials:
    print("Training failures (not included as successful models):", failed_trials)
refresh_download_bundle(run_dir, show_link=True)

## 12. Predictions and errors for every completed model

**Voltage residual** means $e(t)=\hat y(t)-y(t)$ in volts. Positive means our prediction
is too high; negative means too low. Absolute error $|e(t)|$ measures size only.
These are different from the **Duffing physics residual** used in PI-MLP training.

Each model gets a four-panel figure: full-record prediction, test prediction, full-record
signed residual, and test signed residual. All use the already computed continuous free run;
none restart at the test boundary. The original CSV row and time are exported for every sample.

The maps put **time on the horizontal axis and models on the vertical axis**. Signed-error
maps show direction; absolute-error maps show magnitude. Full-record and test maps use the
same color limits across every model and both intervals, with no percentile clipping.
Gray means a non-finite prediction. A divergent trajectory is marked as failed in the scores,
even if its early samples were finite. Failed training trials have no completed-model plots.

A separate plot shows trailing 64-sample RMSE for each model. Each train/test interval
starts with 63 blank samples so the window never crosses the boundary. This is a local
diagnostic; the comparison table still uses the assignment's whole-interval RMSE.

In [ ]:
model_names = [trial["name"] for trial in experiment_plan if trial["name"] in evaluation_results]
if not model_names:
    raise ValueError("Evaluate at least one completed model before plotting.")

def save_and_show(fig, filename):
    fig.savefig(run_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# One learning-history row for EVERY model, with data and physics losses kept separate.
fig, axes = plt.subplots(len(model_names), 2, figsize=(14, 2.8 * len(model_names)), squeeze=False)
for row, name in enumerate(model_names):
    history = training_histories[name]
    axes[row, 0].plot(history["update"], history["section_rmse_V"], lw=0.8)
    axes[row, 0].set(title=name, ylabel="Section RMSE [V]")
    for field, label in [("normalized_data_loss", "Data"), ("weighted_physics_loss", "Weighted physics"),
                         ("normalized_loss", "Total")]:
        axes[row, 1].plot(history["update"], history[field], lw=0.8, label=label)
    axes[row, 1].set(title="Training objective", ylabel="Dimensionless loss")
    axes[row, 1].legend()
    for axis in axes[row]:
        axis.set_xlabel("Weight update")
        axis.grid(alpha=0.3)
fig.tight_layout()
save_and_show(fig, "all_models_training_history.png")

for name in model_names:
    table = prediction_tables[name]
    fig, axes = plt.subplots(2, 2, figsize=(15, 7))
    for column, (section, label) in enumerate([(slice(None), "Full excerpt"), (slice(training_sample_count, None), "Test interval")]):
        part = table.iloc[section]
        prediction = part["predicted_output_V"].replace([np.inf, -np.inf], np.nan)
        axes[0, column].plot(part["time_s"], part["measured_output_V"], lw=0.8, label="Measured")
        axes[0, column].plot(part["time_s"], prediction, lw=0.8, alpha=0.8, label="Simulated")
        axes[0, column].set(title=label, ylabel="Output voltage [V]")
        axes[0, column].legend()
        axes[1, column].plot(part["time_s"], part["residual_V"].replace([np.inf, -np.inf], np.nan), lw=0.8)
        axes[1, column].axhline(0, color="black", lw=0.7)
        axes[1, column].set(ylabel="Predicted - measured [V]", xlabel="Time since excerpt start [s]")
    for axis in axes[:, 0]:
        axis.axvline(t_test[0], color="black", ls="--", lw=0.8)
    for axis in axes.flat:
        axis.grid(alpha=0.3)
    status = "" if evaluation_results[name]["finite_trajectory"] else " — FAILED: non-finite trajectory"
    fig.suptitle(f"{name}: test RMSE {evaluation_results[name]['test_free_run_rmse_V']:.6f} V{status}")
    fig.tight_layout()
    save_and_show(fig, f"{name}_prediction_and_residual.png")

errors = np.vstack([prediction_tables[name]["residual_V"].to_numpy() for name in model_names])
finite_values = np.abs(errors[np.isfinite(errors)])
color_limit = max(float(finite_values.max()), 1e-12) if finite_values.size else 1.0
fig, axes = plt.subplots(2, 2, figsize=(16, max(6, 1.0 * len(model_names))), layout="constrained")
for row, (absolute, label, palette) in enumerate([(False, "Signed error", "RdBu_r"), (True, "Absolute error", "magma")]):
    cmap = plt.get_cmap(palette).copy()
    cmap.set_bad("lightgray")
    for column, (start, label_time) in enumerate([(0, "Full excerpt"), (training_sample_count, "Test interval")]):
        values = errors[:, start:]
        values = np.abs(values) if absolute else values
        im = axes[row, column].imshow(np.ma.masked_invalid(values), aspect="auto", interpolation="nearest",
            origin="upper", cmap=cmap, vmin=0 if absolute else -color_limit, vmax=color_limit,
            extent=[t[start] - dt/2, t[-1] + dt/2, len(model_names)-0.5, -0.5])
        axes[row, column].set_yticks(np.arange(len(model_names)), labels=model_names)
        axes[row, column].set(title=f"{label}: {label_time}", xlabel="Time since excerpt start [s]")
        if start == 0:
            axes[row, column].axvline(t_test[0], color="limegreen", ls="--", lw=1)
    fig.colorbar(im, ax=axes[row, :].tolist(), label=f"{label} [V]", shrink=0.85)
save_and_show(fig, "all_models_residual_maps.png")

fig, axes = plt.subplots(1, 2, figsize=(15, 4), layout="constrained")
for name in model_names:
    table = prediction_tables[name]
    for axis, start in zip(axes, [0, training_sample_count]):
        axis.plot(t[start:], table["rolling_64_sample_rmse_V"].iloc[start:], lw=0.9, label=name)
for axis, title in zip(axes, ["Full excerpt", "Test interval"]):
    axis.set(title=title, xlabel="Time since excerpt start [s]", ylabel="Trailing 64-sample RMSE [V]")
    axis.grid(alpha=0.3)
axes[0].axvline(t_test[0], color="black", ls="--", lw=0.8)
axes[1].legend(fontsize=8)
save_and_show(fig, "all_models_local_rmse.png")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
x = np.arange(len(architecture_configs))
for axis, metric, label in zip(axes, ["train_free_run_rmse_V", "test_free_run_rmse_V"], ["Training", "Test"]):
    scores = comparison.pivot(index="architecture", columns="family", values=metric).reindex(
        index=list(architecture_configs), columns=["MLP", "PI-MLP"])
    for offset, family in [(-0.18, "MLP"), (0.18, "PI-MLP")]:
        values = scores[family].to_numpy()
        axis.bar(x+offset, np.where(np.isfinite(values), values, np.nan), width=0.36, label=family)
        for position, value in zip(x+offset, values):
            if not np.isfinite(value):
                axis.text(position, 0, "missing/failed", rotation=90, va="bottom", ha="center", fontsize=7)
    if label == "Test":
        axis.axhline(comparison["zero_test_baseline_V"].iloc[0], ls="--", color="black", label="Zero baseline")
    axis.set_xticks(x, labels=list(architecture_configs))
    axis.set(title=f"{label} continuous free-run RMSE", ylabel="RMSE [V]")
    axis.legend()
save_and_show(fig, "MLP_vs_PI_MLP_rmse.png")
refresh_download_bundle(run_dir, show_link=True)

## 13. Before ending the Kaggle session

The last cell below lists checkpoints and displays the final ZIP download link again.
**Download that ZIP and keep it on your computer.** Open it and check that it contains
`*_final.pt` files for the completed models. A notebook download by itself does not include them.
If using Save & Run All, also check the completed version's Output tab for the ZIP.

The ZIP includes periodic/final checkpoints, histories, settings, file hashes, comparison
tables, all per-model figures, residual maps, and per-timestamp error CSVs (once evaluation
and plotting cells have run). `all_models_residuals.csv` is the easiest table to inspect a
specific timestamp across networks. `paired_comparison.csv` compares physics-informed and
plain models with the same architecture.

To load a model **without retraining**, run the setup/definition cells, skip the training
execution cell, and use this example with your real path:

```python
loaded_model, saved = restore_trial_for_prediction("/path/to/B_64_64_MLP_adam_final.pt")
# Use saved['preprocessing'] for its scales, centering, dt and initial state.
```

To recreate all evaluation plots without new updates, set `RESUME_RUN_DIR` to a fully
completed run and run the training execution cell: it loads and skips every completed
trial. If the saved run is incomplete, that route resumes unfinished trials.

Remaining limits: noisy rate estimates, short training sections versus a long free run,
one seed, and an untuned physics weight. A low physics residual does not prove low voltage
error or uniquely identified coefficients. This notebook's new experiment has not been
trained during editing; do not interpret the new plots or scores as already measured.

In [ ]:
checkpoint_inventory = pd.read_csv(run_dir / "checkpoint_inventory.csv")
display(checkpoint_inventory)
expected_final = {f"{trial['name']}_final.pt" for trial in experiment_plan}
present_final = {p.name for p in run_dir.glob("*_final.pt")}
missing_final = sorted(expected_final - present_final)
print(f"Completed checkpoints: {len(expected_final & present_final)}/{len(expected_final)}")
if missing_final:
    print("Not completed:", missing_final)
final_archive = refresh_download_bundle(run_dir, show_link=True)
print("Keep the downloaded ZIP. This .ipynb alone does not contain the learned weights.")